# Performance Profiling — Credit Scoring Model

This notebook profiles the prediction pipeline to identify bottlenecks and evaluate optimization options.

## Contents
1. Baseline latency measurement (LightGBM native)
2. cProfile breakdown of the prediction pipeline
3. ONNX Runtime conversion and benchmark
4. Batch size scaling analysis

In [1]:
import sys
sys.path.insert(0, "..")

import time
import numpy as np
import pandas as pd
from app.model import load_model, predict, predict_batch
from app.features import FEATURE_NAMES

model = load_model()
print(f"Model loaded: {model.num_feature()} features")

Model loaded: 795 features


## 1. Baseline Latency

In [2]:
# Single prediction latency (1000 iterations)
features = {name: None for name in FEATURE_NAMES}
features["AMT_INCOME_TOTAL"] = 202500.0
features["AMT_CREDIT"] = 406597.5
features["EXT_SOURCE_2"] = 0.263

latencies = []
for _ in range(1000):
    start = time.perf_counter()
    predict(features)
    latencies.append((time.perf_counter() - start) * 1000)

latencies = np.array(latencies)
print(f"Single prediction latency (ms):")
print(f"  Mean:   {latencies.mean():.2f}")
print(f"  Median: {np.median(latencies):.2f}")
print(f"  P95:    {np.percentile(latencies, 95):.2f}")
print(f"  P99:    {np.percentile(latencies, 99):.2f}")

Single prediction latency (ms):
  Mean:   1.13
  Median: 1.09
  P95:    1.32
  P99:    1.64


## 2. cProfile Analysis

In [3]:
import cProfile
import pstats
from io import StringIO

profiler = cProfile.Profile()
profiler.enable()

for _ in range(100):
    predict(features)

profiler.disable()

stream = StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.sort_stats("cumulative")
stats.print_stats(20)
print(stream.getvalue())

         817924 function calls (815518 primitive calls) in 0.202 seconds

   Ordered by: cumulative time
   List reduced from 345 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      100    0.000    0.000    0.204    0.002 /Users/passkey1510/workspace/data-scientist-ocr/project-8/notebooks/../app/model.py:32(predict)
      3/2    0.000    0.000    0.199    0.100 /Users/passkey1510/workspace/data-scientist-ocr/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3665(run_code)
      3/2    0.000    0.000    0.199    0.100 {built-in method builtins.exec}
        1    0.000    0.000    0.199    0.199 /var/folders/4c/z0lk8svn62nfwqy0v73zxmsh0000gn/T/ipykernel_24445/3780391246.py:1(<module>)
      100    0.003    0.000    0.162    0.002 /Users/passkey1510/workspace/data-scientist-ocr/.venv/lib/python3.12/site-packages/lightgbm/basic.py:4701(predict)
      100    0.001    0.000    0.148    0.001 /Users/passkey1510/workspa

## 3. ONNX Runtime Benchmark (Optional)

If `onnxruntime` and `onnxmltools` are installed, we convert the LightGBM model to ONNX and compare inference speed.

In [4]:
try:
    import onnxruntime as ort
    import onnxmltools
    from onnxmltools.convert import convert_lightgbm
    from onnxconverter_common.data_types import FloatTensorType

    initial_type = [("input", FloatTensorType([None, len(FEATURE_NAMES)]))]
    onnx_model = convert_lightgbm(model, initial_types=initial_type)
    onnxmltools.utils.save_model(onnx_model, "../model/model.onnx")

    session = ort.InferenceSession("../model/model.onnx")
    input_name = session.get_inputs()[0].name

    # Prepare data
    df = pd.DataFrame([features], columns=FEATURE_NAMES).fillna(0).astype(np.float32)
    input_data = df.values

    onnx_latencies = []
    for _ in range(1000):
        start = time.perf_counter()
        session.run(None, {input_name: input_data})
        onnx_latencies.append((time.perf_counter() - start) * 1000)

    onnx_latencies = np.array(onnx_latencies)
    print(f"ONNX Runtime latency (ms):")
    print(f"  Mean:   {onnx_latencies.mean():.2f}")
    print(f"  Median: {np.median(onnx_latencies):.2f}")
    print(f"  P95:    {np.percentile(onnx_latencies, 95):.2f}")
    print(f"\nSpeedup: {latencies.mean() / onnx_latencies.mean():.2f}x")

except ImportError:
    print("onnxruntime / onnxmltools not installed. Skipping ONNX benchmark.")
    print("Install with: pip install onnxruntime onnxmltools")

onnxruntime / onnxmltools not installed. Skipping ONNX benchmark.
Install with: pip install onnxruntime onnxmltools


## 4. Batch Size Scaling

In [5]:
batch_sizes = [1, 10, 50, 100, 500, 1000]
results = []

for n in batch_sizes:
    records = [features] * n
    times = []
    for _ in range(10):
        start = time.perf_counter()
        predict_batch(records)
        times.append((time.perf_counter() - start) * 1000)
    avg_ms = np.mean(times)
    results.append({"batch_size": n, "total_ms": round(avg_ms, 2), "per_record_ms": round(avg_ms / n, 3)})

results_df = pd.DataFrame(results)
print("Batch size scaling:")
results_df

Batch size scaling:


,batch_size,total_ms,per_record_ms
0,1,2.73,2.726
1,10,2.42,0.242
2,50,7.88,0.158
3,100,14.39,0.144
4,500,67.15,0.134
5,1000,132.93,0.133


## Conclusions

Key findings:
- **DataFrame construction** (pandas) typically dominates single-prediction latency
- **LightGBM inference** itself is very fast (<1ms for single predictions)
- **Batch predictions** benefit from vectorization — per-record cost drops significantly
- **ONNX Runtime** may provide modest speedup for inference-heavy workloads

### Optimization Recommendations
1. Use batch endpoints where possible to amortize DataFrame overhead
2. Consider pre-allocating numpy arrays for high-throughput scenarios
3. ONNX conversion is worthwhile only if inference latency dominates (not data prep)